# Lab: Data-Centric vs Model-Centric approaches

This lab gives an introduction to data-centric vs model-centric approaches to machine learning problems, showing how data-centric approaches can outperform purely model-centric approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [1]:
!pip install scikit-learn pandas

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
     ------------- -------------------------- 2.9/8.3 MB 13.9 MB/s eta 0:00:01
     -------------------------- ------------- 5.5/8.3 MB 14.6 MB/s eta 0:00:01
     ---------------------------------------- 8.3/8.3 MB 14.0 MB/s  0:00:00
     ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
     ----------- ---------------------------- 2.9/9.9 MB 14.0 MB/s eta 0:00:01
     ----------------------- ---------------- 5.8/9.9 MB 16.0 MB/s eta 0:00:01
     ------------------------------- -------- 7.9/9.9 MB 13.5 MB/s eta 0:00:01
     ---------------------------------------- 9.9/9.9 MB 14.6 MB/s  0:00:00
     ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
     ---------- ----------------------------- 3.4/12.6 MB 22.3 MB/s eta 0:00:01
     -------------------- ------------------- 6.6/12.6 MB 19.2 MB/s eta 0:00:01
     -------------

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [2]:
import pandas as pd

In [4]:
train = pd.read_csv('reviews_train.csv')
test = pd.read_csv('reviews_test.csv')

test.sample(5)

,review,label
901,Didn't arrive on my device. No customer support.,bad
203,A great magazine at a great price ($5!),good
163,This is a great magazine. I get so many good ...,good
192,"Are you kidding, this is a great magazine",good
471,Every issue is a must-read.,good


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [6]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [7]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [8]:
from sklearn import metrics

In [9]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [10]:
evaluate(sgd_clf)

Accuracy: 76.7%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html). 

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

In [22]:
# YOUR CODE HERE
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

nb_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

_ = nb_clf.fit(train['review'], train['label'])
evaluate(nb_clf)
# evaluate your model and see if it does better
# than the ones we provided

Accuracy: 85.3%


## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [24]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [25]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [ ]:
# YOUR CODE HERE
train.sample(10)
# some special HTML tokens, such as </div> may affect the result and the label may not be aligned.
# The sentence 'Wondeful...loved it<div class="usage">' was labeled as bad which is absolutely wrong.

,review,label
2226,I still haven't gotten the first magazine so h...,bad
5016,</a>This is one of my favorite magazines - col...,bad
1390,</html>Cancelled didn't even know I ordered it.,good
1126,Best educational magazine I truly enjoy and wa...,good
4907,"</div>These magazines all do the same thing, t...",good
2724,This magazine has something for everyone. Ther...,good
1906,<TR>So easy. It auto-renews and it's cheaper t...,bad
4199,</div>This was requested for a gift & he so li...,bad
34,"Wonderful...loved it<div class=""usage"">",bad
3981,"Only magazine I subscribe to, love it and love...",good


## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [31]:
import re
def is_bad_data(review: str) -> bool:
    # YOUR CODE HERE
    if re.search(r"<.*?>", review):
        return True
    return False

## Creating the cleaned training set

In [32]:
train_clean = train[~train['review'].map(is_bad_data)]

## Evaluating a model trained on the clean training set

In [33]:
from sklearn import clone

In [34]:
sgd_clf_clean = clone(sgd_clf)

In [35]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [36]:
evaluate(sgd_clf_clean)

Accuracy: 96.8%
